# GraphConsis & BWGNN — Colab experiments

Train **GraphConsis** and **BWGNN** on YelpChi, Amazon, T-Finance, and T-Social.

Each experiment cell has a parameter dict you can edit. After training, artifacts are written under:

`notebook_runs/<Model>/<Dataset>/seed<seed>_<timestamp>_ep…/`

(config, metrics, split, predictions, checkpoint), then that folder is **zipped**. On Colab the zip downloads automatically.

**Runtime → Change runtime type → GPU** before installing packages.

T-Finance (~42M edges) and T-Social (multi-GB) are heavy. Prefer a GPU High-RAM runtime for those cells.

## 1. Repository on Colab

Opening this notebook from GitHub does **not** copy the rest of the repo. Clone it, point at a Drive folder, or upload a zip. Local Jupyter can skip this if you already opened the notebook from the repo root.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# If the GitHub repo is private, use a tokenized URL or skip clone and set DRIVE_REPO.
REPO_URL = "https://github.com/kavehkarimadini/Graph-Fraud-WK.git"
DRIVE_REPO = ""  # e.g. /content/drive/MyDrive/Graph-Fraud-WK
MOUNT_DRIVE = True  # mount Google Drive (needed for dataset zips)


def _looks_like_repo(path):
    return (Path(path) / "algorithms" / "GraphConsis" / "supervised_train.py").is_file()


if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

repo = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path(DRIVE_REPO)] if DRIVE_REPO else [Path.cwd(), *Path.cwd().parents]:
    if _looks_like_repo(candidate):
        repo = candidate
        break

if repo is None and IN_COLAB:
    target = Path("/content/Graph-Fraud-WK")
    if not _looks_like_repo(target):
        if target.exists():
            shutil.rmtree(target)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    repo = target

if repo is None:
    raise FileNotFoundError(
        "Repo root not found. Clone Graph-Fraud-WK, set DRIVE_REPO, or run this notebook from the repo root."
    )

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Repo root:", repo.resolve())
print("Colab:", IN_COLAB)

## 2. Install packages (Colab)

Colab already ships CUDA PyTorch. Install PyG and sympy only — do **not** `pip install dgl` (broken on many Colab / Windows Python builds). T-Finance / T-Social load without DGL.

Re-run this cell after a runtime restart.

In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

%pip install -q torch-geometric sympy

import torch_geometric
import sympy
print("torch-geometric", torch_geometric.__version__, "sympy", sympy.__version__)

## 3. Datasets from Google Drive

The graphs are **not** in GitHub. Upload the official zips to a Drive folder (any name), then set `DRIVE_DATASET_DIR` to that folder **relative to MyDrive**.

Example: zips in `My Drive / GraphFraud / zips` → `DRIVE_DATASET_DIR = "GraphFraud/zips"`

| Zip on Drive | Extracted into repo `dataset/` |
| --- | --- |
| `YelpChi.zip` | `YelpChi.mat` |
| `Amazon.zip` | `Amazon.mat` |
| `tfinance.zip` | `tfinance` (no extension) |
| `tsocial.zip` | `tsocial` (no extension, ~4 GB) |

The cell mounts Drive if needed, unzips **from Drive into local** `dataset/` (Colab disk), and does not copy the zip files. T-Social takes a while. You can also point at already-extracted files.

In [ ]:
from utils.notebook_runner import (
    dataset_dir,
    ensure_drive_mounted,
    missing_dataset_files,
    prepare_datasets_from_drive,
)

# Folder on Google Drive that holds the zips (or already-extracted files).
# Relative to My Drive after mount, e.g. "GraphFraud/zips"
# Absolute also works: "/content/drive/MyDrive/GraphFraud/zips"
DRIVE_DATASET_DIR = "GraphFraud/zips"

if IN_COLAB:
    ensure_drive_mounted()

ddir = dataset_dir()
if not str(DRIVE_DATASET_DIR).strip():
    raise ValueError("Set DRIVE_DATASET_DIR to your Drive folder (relative to MyDrive).")

missing = prepare_datasets_from_drive(DRIVE_DATASET_DIR, dest=ddir)

print("dataset/:", ddir.resolve())
for p in sorted(ddir.iterdir()):
    if p.is_file() and not p.name.endswith(".zip"):
        print("  {:<24} {:8.1f} MB".format(p.name, p.stat().st_size / 1e6))
if missing:
    print("Still missing:", missing)
    print("Check the Drive path and that the zip names are YelpChi.zip, Amazon.zip, tfinance.zip, tsocial.zip.")
else:
    print("All four dataset files are present.")

## 4. Run helpers

Edit the dict in a cell below, then run that cell. Each run gets its own folder plus a zip.

Set `download=False` if you do not want the Colab download dialog every time (zips are still written next to the folder).

In [ ]:
import torch
from utils.notebook_runner import (
    notebook_runs_dir,
    run_bwgnn,
    run_graphconsis,
    zip_all_notebook_runs,
)

print("Outputs:", notebook_runs_dir().resolve())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

---
## GraphConsis

Tunable keys: `epochs`, `learning_rate`, `dim_1`, `batch_size`, `dropout`, `weight_decay`, `context_dim`, `sigmoid`, `samples_1`, `samples_2`, `test_size`, `seed`, `gpu`.

Folder: `notebook_runs/GraphConsis/<Dataset>/seed<seed>_<tag>/`

In [ ]:
# GraphConsis · YelpChi — edit params, then run this cell
GC_YELPCHI = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.2,
    seed=123,
    gpu="0",
)

gc_yelpchi = run_graphconsis("yelpchi", download=True, **GC_YELPCHI)
gc_yelpchi["metrics"]

In [ ]:
# GraphConsis · Amazon — edit params, then run this cell
GC_AMAZON = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_amazon = run_graphconsis("amazon", download=True, **GC_AMAZON)
gc_amazon["metrics"]

In [ ]:
# GraphConsis · T-Finance — 42M edges; GPU recommended
GC_TFINANCE = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_tfinance = run_graphconsis("tfinance", download=True, **GC_TFINANCE)
gc_tfinance["metrics"]

In [ ]:
# GraphConsis · T-Social — very large; smaller batch / fewer epochs by default
GC_TSOCIAL = dict(
    epochs=20,
    learning_rate=0.01,
    dim_1=64,
    batch_size=256,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=10,
    samples_2=5,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_tsocial = run_graphconsis("tsocial", download=True, **GC_TSOCIAL)
gc_tsocial["metrics"]

---
## BWGNN

Tunable keys: `epochs`, `learning_rate`, `hid_dim`, `order`, `weight_decay`, `train_ratio` (overrides `test_size`), `test_size`, `seed`, `gpu`.

Paper defaults: `train_ratio=0.4`; T-Social uses `hid_dim=10`, `order=5`.

Folder: `notebook_runs/BWGNN/<dataset>/seed<seed>_<tag>/`

In [ ]:
# BWGNN · YelpChi — edit params, then run this cell
BW_YELPCHI = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=None,
    test_size=0.2,
    seed=123,
    gpu="0",
)

bw_yelpchi = run_bwgnn("yelpchi", download=True, **BW_YELPCHI)
bw_yelpchi["metrics"]

In [ ]:
# BWGNN · Amazon — edit params, then run this cell
BW_AMAZON = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_amazon = run_bwgnn("amazon", download=True, **BW_AMAZON)
bw_amazon["metrics"]

In [ ]:
# BWGNN · T-Finance — paper: hid 64, order 2, 40% train
BW_TFINANCE = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_tfinance = run_bwgnn("tfinance", download=True, **BW_TFINANCE)
bw_tfinance["metrics"]

In [ ]:
# BWGNN · T-Social — paper: hid 10, order 5; needs a large GPU
BW_TSOCIAL = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=10,
    order=5,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_tsocial = run_bwgnn("tsocial", download=True, **BW_TSOCIAL)
bw_tsocial["metrics"]

## Download every notebook run

Optional: one zip of `notebook_runs/` after you have finished several cells.

In [ ]:
zip_all_notebook_runs(download=True)